In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
BASE_PATH = "/Volumes/main/lakehouse_marketing"
RAW_PATH = f"{BASE_PATH}/raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"

In [0]:
campaign_schema = StructType([
    StructField("campaign_id", StringType(), True),
    StructField("campaign_name", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True)
])

df_campaigns_raw = spark.read\
                    .schema(campaign_schema)\
                    .option("header", "true")\
                    .csv(f"{RAW_PATH}/campaigns")

In [0]:
df_campaigns_bronze = df_campaigns_raw\
                        .withColumn("ingestion_timestamp", F.current_timestamp())\
                        .withColumn("source_file", F.col("_metadata.file_path"))

* Escrita na Bronze

In [0]:
BRONZE_CAMPAIGNS_PATH = f"{BRONZE_PATH}/campaigns"

df_campaigns_bronze.write\
    .format('delta')\
    .mode('overwrite')\
    .save(BRONZE_CAMPAIGNS_PATH)

* Validação

In [0]:
spark.read\
    .format('delta')\
    .load("/Volumes/main/lakehouse_marketing/bronze/campaigns/")\
    .count()


In [0]:
# display(spark.read\
#     .format("delta")\
#     .load("/Volumes/main/lakehouse_marketing/bronze/campaigns/"))